## Setup and Imports

In [29]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [30]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [31]:
import json
import os
import random
import numpy as np
import torch
import torch.nn as nn
from transformers import (
    AutoTokenizer,
    AutoModel,
    TrainingArguments,
    Trainer
)
from torch.utils.data import Dataset
from tqdm import tqdm
from collections import Counter

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Setup complete")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Setup complete
PyTorch version: 2.11.0.dev20260204+cu128
CUDA available: True


### Define Relation Labels and Configuration

This section defines the legal entity categories, relation predicates, and core configuration for the DeBERTa-v3-large training run.

Key choices in this baseline:
- **Model:** `microsoft/deberta-v3-large` — stronger encoder than BERT-base, consistently outperforms it on RE tasks
- **Negative multiplier:** 5× — same as the best-performing BERT run (A0); higher recall on rare relation types
- **All collections included:** gold + silver + silver_2025 + bronze — bronze is kept because it improves recall on rare predicates

In [32]:
import re

LEGAL_ENTITY_LABELS = {
    "anatomical location","animal","bacteria","biomedical technique","chemical","DDF",
    "dietary supplement","drug","food","gene","human","microbiome","statistical technique"
}
LEGAL_RELATION_LABELS = {
    "administered","affect","change abundance","change effect","change expression","compared to",
    "impact","influence","interact","is a","is linked to","located in","part of","produced by",
    "strike","target","used by"
}

def norm_ent(label: str) -> str:
    if label is None:
        return ""
    lab = str(label).strip()
    if lab.lower() == "ddf":
        return "DDF"
    return lab

def norm_span(s: str) -> str:
    # consigliato per ridurre mismatch banali sugli span
    s = str(s).strip()
    s = re.sub(r"\s+", " ", s)
    return s

### Define Legal Entity and Relation Labels

Defines the allowed entity types and relation predicates from the GutBrainIE ontology.

These sets are used to:
- validate dataset annotations
- filter invalid entity–predicate–entity triples during example preparation
- restrict negative sampling to legal entity type pairs only

In [33]:
# Define legal relation predicates
RELATION_LABELS = [
    "no relation",  # For negative samples
    "administered",
    "affect",
    "change abundance",
    "change effect",
    "change expression",
    "compared to",
    "impact",
    "influence",
    "interact",
    "is a",
    "is linked to",
    "located in",
    "part of",
    "produced by",
    "strike",
    "target",
    "used by"
]

label2id = {label: idx for idx, label in enumerate(RELATION_LABELS)}
id2label = {idx: label for idx, label in enumerate(RELATION_LABELS)}

print(f"Total relation labels: {len(RELATION_LABELS)}")
print(f"Labels: {RELATION_LABELS}")

# Define legal entity type relations (subject_label, predicate, object_label)
# Order matters: relation is from subject to object
LEGAL_RELATIONS = [
    ("DDF", "affect", "DDF"),
    ("microbiome", "is linked to", "DDF"),
    ("DDF", "target", "human"),
    ("drug", "change effect", "DDF"),
    ("DDF", "is a", "DDF"),
    ("microbiome", "located in", "human"),
    ("chemical", "influence", "DDF"),
    ("dietary supplement", "influence", "DDF"),
    ("DDF", "target", "animal"),
    ("chemical", "impact", "microbiome"),
    ("anatomical location", "located in", "animal"),
    ("microbiome", "located in", "animal"),
    ("chemical", "located in", "anatomical location"),
    ("bacteria", "part of", "microbiome"),
    ("DDF", "strike", "anatomical location"),
    ("drug", "administered", "animal"),
    ("bacteria", "influence", "DDF"),
    ("drug", "impact", "microbiome"),
    ("DDF", "change abundance", "microbiome"),
    ("microbiome", "located in", "anatomical location"),
    ("microbiome", "used by", "biomedical technique"),
    ("chemical", "produced by", "microbiome"),
    ("dietary supplement", "impact", "microbiome"),
    ("bacteria", "located in", "animal"),
    ("animal", "used by", "biomedical technique"),
    ("chemical", "impact", "bacteria"),
    ("chemical", "located in", "animal"),
    ("food", "impact", "bacteria"),
    ("microbiome", "compared to", "microbiome"),
    ("human", "used by", "biomedical technique"),
    ("bacteria", "change expression", "gene"),
    ("chemical", "located in", "human"),
    ("drug", "interact", "chemical"),
    ("food", "administered", "human"),
    ("DDF", "change abundance", "bacteria"),
    ("chemical", "interact", "chemical"),
    ("chemical", "part of", "chemical"),
    ("dietary supplement", "impact", "bacteria"),
    ("DDF", "interact", "chemical"),
    ("food", "impact", "microbiome"),
    ("food", "influence", "DDF"),
    ("bacteria", "located in", "human"),
    ("dietary supplement", "administered", "human"),
    ("bacteria", "interact", "chemical"),
    ("drug", "change expression", "gene"),
    ("drug", "impact", "bacteria"),
    ("drug", "administered", "human"),
    ("anatomical location", "located in", "human"),
    ("dietary supplement", "change expression", "gene"),
    ("chemical", "change expression", "gene"),
    ("bacteria", "interact", "bacteria"),
    ("drug", "interact", "drug"),
    ("microbiome", "change expression", "gene"),
    ("bacteria", "interact", "drug"),
    ("food", "change expression", "gene")
]

# Create lookup structures for legal relations
# Map (subject_label, object_label) -> set of predicates
legal_pairs = {}
for s, p, o in LEGAL_RELATIONS:
    s = norm_ent(s); o = norm_ent(o)
    legal_pairs.setdefault((s, o), set()).add(p)

print(f"\nTotal legal relation patterns: {len(LEGAL_RELATIONS)}")
print(f"Total unique entity type pairs: {len(legal_pairs)}")

# Configuration
model_name = "microsoft/deberta-v3-base"
output_model_dir = "../../models/deberta_v3_base_re_baseline"
max_length = 256
NEGATIVE_SAMPLE_MULTIPLIER = 5  # Number of negative samples per positive sample

print(f"\nModel: {model_name}")
print(f"Output directory: {output_model_dir}")
print(f"Negative sample multiplier: {NEGATIVE_SAMPLE_MULTIPLIER}")

Total relation labels: 18
Labels: ['no relation', 'administered', 'affect', 'change abundance', 'change effect', 'change expression', 'compared to', 'impact', 'influence', 'interact', 'is a', 'is linked to', 'located in', 'part of', 'produced by', 'strike', 'target', 'used by']

Total legal relation patterns: 55
Total unique entity type pairs: 52

Model: microsoft/deberta-v3-base
Output directory: ../../models/deberta_v3_base_re_baseline
Negative sample multiplier: 5


### DeBERTa-v3 Model with Typed Entity Markers and Mention-Mean Pooling

Custom PyTorch model built on top of `deberta-v3-large` for mention-level relation extraction.

**Key differences from the BERT baseline:**

1. **No `token_type_ids`** — DeBERTa-v3 does not use token type IDs; passing them causes a silent error.

2. **Typed entity markers** — instead of generic `[E1]`/`[E2]`, the input uses type-specific markers:
   - `[CHEMICAL]α-SMA[/CHEMICAL]` for the subject
   - `[ANATOMICAL_LOCATION]colon[/ANATOMICAL_LOCATION]` for the object

   This encodes entity type information directly inside the transformer context, not only in the classification head.

3. **Mention-mean pooling** — the entity representation is the mean of all token embeddings within the marked span, rather than just the single marker token. More robust for multi-token biomedical entities (e.g. *Lactobacillus rhamnosus GG*).

4. **Sample weight support** — each training example carries a quality weight (gold=1.0, silver=0.8, silver_2025=0.7, bronze=0.4). The loss is computed as a weighted mean using `CrossEntropyLoss(reduction="none")`.

The two entity representations are concatenated and passed through a linear classification head to predict the relation predicate.

In [34]:
def entity_average(hidden, mask):
    """
    Mean pooling over tokens belonging to the entity mention.
    """
    mask = mask.unsqueeze(-1).float()
    summed = (hidden * mask).sum(dim=1)
    count = mask.sum(dim=1).clamp(min=1e-6)
    return summed / count

In [35]:
class DeBERTaForREWithEntityMarkers(nn.Module):

    def __init__(self, model_name, num_labels):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.1)
        hidden_size = self.bert.config.hidden_size
        self.classifier = nn.Linear(hidden_size * 2, num_labels)
        self.num_labels = num_labels

    def gradient_checkpointing_enable(self, gradient_checkpointing_kwargs=None):
        self.bert.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs=gradient_checkpointing_kwargs
        )

    def gradient_checkpointing_disable(self):
        self.bert.gradient_checkpointing_disable()

    def forward(self, input_ids, attention_mask, e1_mask, e2_mask,
                labels=None, sample_weight=None, **kwargs):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        sequence_output = outputs.last_hidden_state
        e1_h = entity_average(sequence_output, e1_mask)
        e2_h = entity_average(sequence_output, e2_mask)
        concat_h = torch.cat([e1_h, e2_h], dim=-1)
        concat_h = self.dropout(concat_h)
        logits = self.classifier(concat_h)

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss(reduction="none")
            per_example_loss = loss_fct(logits, labels)
            if sample_weight is not None:
                loss = (per_example_loss * sample_weight).mean()
            else:
                loss = per_example_loss.mean()

        return {"loss": loss, "logits": logits}


print("DeBERTa RE model class defined")

DeBERTa RE model class defined


## Data Loading Functions

Loads documents from JSON files and attaches source quality metadata to each document.

Quality weights used during training loss:

| Collection | Weight |
|---|---|
| gold | 1.0 |
| silver | 0.8 |
| silver_2025 | 0.7 |
| bronze | 0.4 |

Conservative weights: the bronze collection is downweighted but not excluded, preserving recall on rare relation types.

In [36]:
def infer_source_quality(path: str) -> str:
    p = path.lower()
    if "gold" in p:      return "gold"
    if "silver_2025" in p: return "silver_2025"
    if "silver" in p:    return "silver"
    if "bronze" in p:    return "bronze"
    return "unknown"

# Pesi conservativi: differenziano qualità senza penalizzare bronze
QUALITY_WEIGHTS = {
    "gold":        1.0,
    "silver":      0.8,
    "silver_2025": 0.7,
    "bronze":      0.4,
    "unknown":     0.5,
}

def load_re_data(file_paths):
    """Load RE data attaching source quality metadata per document."""
    all_data = {}
    for file_path in file_paths:
        if os.path.exists(file_path):
            with open(file_path, "r", encoding="utf-8") as f:
                data = json.load(f)
            quality = infer_source_quality(file_path)
            for pmid, article in data.items():
                article = dict(article)
                article["_source_quality"] = quality
                all_data[pmid] = article
            print(f"Loaded {len(data)} documents from {os.path.basename(file_path)} [{quality}]")
        else:
            print(f"Warning: {file_path} not found")
    return all_data

print("Data loading function defined")


Data loading function defined


## Load Training and Dev Data

In [37]:
# Load training data from three quality levels
train_files = [
    "../../../data/GutBrainIE_Full_Collection_2026/Annotations/Train/gold_quality/json_format/train_gold.json",
    "../../../data/GutBrainIE_Full_Collection_2026/Annotations/Train/silver_quality/json_format/train_silver.json",
    "../../../data/GutBrainIE_Full_Collection_2026/Annotations/Train/bronze_quality/json_format/train_bronze.json",
    "../../../data/GutBrainIE_Full_Collection_2026/Annotations/Train/silver_quality/json_format/train_silver_2025.json",  # incluso
]
#todo change also in other files
train_data = load_re_data(train_files)
print(f"\nTotal training documents: {len(train_data)}")

Loaded 639 documents from train_gold.json [gold]
Loaded 811 documents from train_silver.json [silver]
Loaded 2972 documents from train_bronze.json [bronze]
Loaded 499 documents from train_silver_2025.json [silver_2025]

Total training documents: 4921


In [38]:
# Load dev data
dev_data = load_re_data([
    "../../../data/GutBrainIE_Full_Collection_2026/Annotations/Dev/json_format/dev.json"
])
print(f"Total dev documents: {len(dev_data)}")

Loaded 80 documents from dev.json [unknown]
Total dev documents: 80


## Prepare Relation Extraction Examples

For each document:
1. Extract **positive** examples from gold annotations — one example per annotated relation triple
2. Generate **negative** examples by sampling entity pairs with no annotated relation (5× the positive count)
3. Attach `source_quality` and `sample_weight` to every example for quality-aware loss weighting
4. Restrict negative candidates to legal entity type pairs and pairs within 400 chars of each other

In [39]:
from collections import defaultdict
def create_full_text_with_offsets(title, abstract):
    """
    Create full text by concatenating title and abstract.
    Returns full text and offset for abstract entities.
    """
    full_text = f"{title} {abstract}"
    abstract_offset = len(title) + 1
    return full_text, abstract_offset


def adjust_entity_positions(entity, abstract_offset):
    """
    Adjust entity character positions to account for title + abstract concatenation.
    """
    if entity['location'] == 'abstract':
        return {
            'start_idx': entity['start_idx'] + abstract_offset,
            'end_idx': entity['end_idx'] + abstract_offset,
            'text_span': entity['text_span'],
            'label': entity['label']
        }
    else:
        return {
            'start_idx': entity['start_idx'],
            'end_idx': entity['end_idx'],
            'text_span': entity['text_span'],
            'label': entity['label']
        }


MAX_PAIR_CHARS = 400  # prova 300/400/500

def char_distance(a, b):
    # distanza tra due mention (start inclusive)
    return abs(a["start_idx"] - b["start_idx"])
def prepare_re_examples(data, negative_multiplier=1, legal_pairs=None):
    """
    Prepare relation extraction examples with positive and negative samples.
    Only considers entity pairs that match legal relation patterns.

    Args:
        data: Dictionary of documents with entities and mention_level_relations
        negative_multiplier: Number of negative samples per positive sample
        legal_pairs: Dict mapping (subject_label, object_label) -> set(predicates)

    Returns:
        List of examples: {text, subject, object, predicate, pmid}
    """
    def loc_rank(loc: str) -> int:
        return 0 if loc == "title" else 1  # title preferred over abstract

    def best_pair(subj_cands, obj_cands):
        """
        Choose the best (subject, object) mention pair among duplicates.
        Preference:
          1) title-title > title-abstract > abstract-abstract
          2) same location preferred
          3) minimal distance in text
        """
        best = None
        best_score = None

        for s in subj_cands:
            for o in obj_cands:
                # avoid identical mention used as both
                if s["start_idx"] == o["start_idx"] and s["end_idx"] == o["end_idx"] and s["location"] == o["location"]:
                    continue

                loc_combo = loc_rank(s["location"]) + loc_rank(o["location"])
                same_loc = 0 if s["location"] == o["location"] else 1
                dist = abs(s["start_idx"] - o["start_idx"])
                score = (loc_combo, same_loc, dist)

                if best_score is None or score < best_score:
                    best_score = score
                    best = (s, o)

        return best

    examples = []

    for pmid, article in tqdm(data.items(), desc="Preparing RE examples"):
        title = article["metadata"]["title"]
        abstract = article["metadata"]["abstract"]
        full_text, abstract_offset = create_full_text_with_offsets(title, abstract)

        entities = article["entities"]
        relations = article.get("mention_level_relations", [])

        # normalize + adjust offsets
        adjusted_entities = [
            {
                **adjust_entity_positions(e, abstract_offset),
                "label": norm_ent(e["label"]),
                "text_span": norm_span(e["text_span"]),
                "location": e["location"],
            }
            for e in entities
        ]

        # index for (span,label) -> list of mentions
        ent_index = defaultdict(list)
        for e in adjusted_entities:
            ent_index[(e["text_span"], e["label"])].append(e)

        # -------- positives --------
        positive_pairs = set()

        for relation in relations:
            subj_text = norm_span(relation["subject_text_span"])
            obj_text  = norm_span(relation["object_text_span"])
            subj_lab  = norm_ent(relation["subject_label"])
            obj_lab   = norm_ent(relation["object_label"])
            pred      = relation["predicate"].strip()

            if pred not in LEGAL_RELATION_LABELS:
                continue
            if subj_lab not in LEGAL_ENTITY_LABELS or obj_lab not in LEGAL_ENTITY_LABELS:
                continue

            subj_cands = ent_index.get((subj_text, subj_lab), [])
            obj_cands  = ent_index.get((obj_text, obj_lab), [])

            pair = best_pair(subj_cands, obj_cands)
            if not pair:
                continue

            subject, obj = pair

            # optional safety: keep only legal type-pairs if provided
            type_pair = (subject["label"], obj["label"])
            if legal_pairs is not None and type_pair not in legal_pairs:
                continue

            sq = article.get("_source_quality", "unknown")
            examples.append({
                "text": full_text,
                "subject": subject,
                "object": obj,
                "predicate": pred,
                "pmid": pmid,
                "source_quality": sq,
                "sample_weight": QUALITY_WEIGHTS.get(sq, 0.5),
            })

            pair_key = (subject["start_idx"], subject["end_idx"], obj["start_idx"], obj["end_idx"])
            positive_pairs.add(pair_key)

        # -------- negatives --------
        num_negatives = len(positive_pairs) * negative_multiplier
        negative_candidates = []

        if num_negatives > 0:
            for i, subj in enumerate(adjusted_entities):
                for j, obj in enumerate(adjusted_entities):
                    if i == j:
                        continue

                    type_pair = (subj["label"], obj["label"])
                    if legal_pairs is not None and type_pair not in legal_pairs:
                        continue
                    if char_distance(subj, obj) > MAX_PAIR_CHARS:
                        continue
                    pair_key = (subj["start_idx"], subj["end_idx"], obj["start_idx"], obj["end_idx"])
                    if pair_key in positive_pairs:
                        continue

                    sq = article.get("_source_quality", "unknown")
                    negative_candidates.append({
                        "text": full_text,
                        "subject": subj,
                        "object": obj,
                        "predicate": "no relation",
                        "pmid": pmid,
                        "source_quality": sq,
                        "sample_weight": QUALITY_WEIGHTS.get(sq, 0.5),
                    })

            if negative_candidates:
                num_to_sample = min(num_negatives, len(negative_candidates))
                examples.extend(random.sample(negative_candidates, num_to_sample))

    return examples


In [40]:
# Prepare training examples
print("Preparing training examples...")
train_examples = prepare_re_examples(train_data, negative_multiplier=NEGATIVE_SAMPLE_MULTIPLIER, legal_pairs=legal_pairs)

# Count positive vs negative
positive_count = sum(1 for ex in train_examples if ex['predicate'] != 'no relation')
negative_count = sum(1 for ex in train_examples if ex['predicate'] == 'no relation')

print(f"\nTraining examples prepared: {len(train_examples)}")
print(f"  Positive examples: {positive_count}")
print(f"  Negative examples: {negative_count}")
print(f"  Ratio (neg/pos): {negative_count/positive_count:.2f}")

Preparing training examples...


Preparing RE examples: 100%|██████████| 4921/4921 [00:04<00:00, 984.59it/s] 



Training examples prepared: 319650
  Positive examples: 53791
  Negative examples: 265859
  Ratio (neg/pos): 4.94


In [41]:
# Prepare dev examples
print("Preparing dev examples...")
dev_examples = prepare_re_examples(dev_data, negative_multiplier=NEGATIVE_SAMPLE_MULTIPLIER, legal_pairs=legal_pairs)

positive_count_dev = sum(1 for ex in dev_examples if ex['predicate'] != 'no relation')
negative_count_dev = sum(1 for ex in dev_examples if ex['predicate'] == 'no relation')

print(f"\nDev examples prepared: {len(dev_examples)}")
print(f"  Positive examples: {positive_count_dev}")
print(f"  Negative examples: {negative_count_dev}")

Preparing dev examples...


Preparing RE examples: 100%|██████████| 80/80 [00:00<00:00, 811.88it/s]


Dev examples prepared: 6580
  Positive examples: 1116
  Negative examples: 5464


In [42]:
# Show example
print("\nExample training instance:")
example = train_examples[0]
print(f"  Text: {example['text'][:150]}...")
print(f"  Subject: '{example['subject']['text_span']}' [{example['subject']['label']}]")
print(f"  Object: '{example['object']['text_span']}' [{example['object']['label']}]")
print(f"  Predicate: {example['predicate']}")


Example training instance:
  Text: Probiotics and microbial metabolites maintain barrier and neuromuscular functions and clean protein aggregation to delay disease progression in TDP43 ...
  Subject: 'α-SMA' [chemical]
  Object: 'colon' [anatomical location]
  Predicate: located in


In [43]:
print("train_docs:", len(train_data))
print("dev_docs:", len(dev_data))

print("train_examples:", len(train_examples))
print("dev_examples:", len(dev_examples))

pos = sum(1 for ex in train_examples if ex["predicate"] != "no relation")
neg = len(train_examples) - pos
print("train_pos:", pos, "train_neg:", neg, "neg/pos:", neg/max(pos,1))

train_docs: 4921
dev_docs: 80
train_examples: 319650
dev_examples: 6580
train_pos: 53791 train_neg: 265859 neg/pos: 4.9424439032551915


## Initialize Tokenizer and Add Typed Entity Markers

Loads the DeBERTa-v3-large tokenizer and adds typed special tokens for all 15 GutBrainIE entity types:
```
[BACTERIA] / [/BACTERIA]
[CHEMICAL] / [/CHEMICAL]
[DDF] / [/DDF]
[ANATOMICAL_LOCATION] / [/ANATOMICAL_LOCATION]
... (one pair per entity type)
```

Generic `[E1]`/`[E2]` markers are also added as fallback for entities without a label.

The typed markers allow the model to distinguish entity types at the transformer level — the type information is available to every attention layer, not only to the final classification head.

In [44]:
print("Initializing tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

# Typed entity markers: un marker per tipo di entità
# Il tipo entra direttamente nel contesto del transformer
ENTITY_TYPES = [
    "anatomical location", "animal", "bacteria", "biomedical technique",
    "chemical", "DDF", "dietary supplement", "drug", "food", "gene",
    "human", "microbiome", "statistical technique",
]

def type_to_marker(label: str) -> str:
    return "[" + label.upper().replace(" ", "_") + "]"

def type_to_end_marker(label: str) -> str:
    return "[/" + label.upper().replace(" ", "_") + "]"

typed_special_tokens = []
for t in ENTITY_TYPES:
    typed_special_tokens.append(type_to_marker(t))
    typed_special_tokens.append(type_to_end_marker(t))

# Aggiungi anche i marker generici come fallback
generic_tokens = ["[E1]", "[/E1]", "[E2]", "[/E2]"]
all_special = typed_special_tokens + generic_tokens

tokenizer.add_special_tokens({"additional_special_tokens": all_special})

# IDs dei marker generici (usati dall'encoder per trovare le posizioni)
e1_token_id     = tokenizer.convert_tokens_to_ids("[E1]")
e1_end_token_id = tokenizer.convert_tokens_to_ids("[/E1]")
e2_token_id     = tokenizer.convert_tokens_to_ids("[E2]")
e2_end_token_id = tokenizer.convert_tokens_to_ids("[/E2]")

print(f"Tokenizer: {tokenizer.__class__.__name__}")
print(f"  Vocab size (with special tokens): {len(tokenizer)}")
print(f"  Typed markers added: {len(typed_special_tokens)}")
print(f"  [E1] token ID: {e1_token_id}")
print(f"  [E2] token ID: {e2_token_id}")


Initializing tokenizer...
Tokenizer: DebertaV2Tokenizer
  Vocab size (with special tokens): 128031
  Typed markers added: 26
  [E1] token ID: 128027
  [E2] token ID: 128029


## Tokenization with Typed Entity Markers

`tokenize_re_example` performs the following steps:

1. **Window extraction** — crops a ±300-char context window around the two entities to avoid truncation of marker tokens
2. **Typed marker insertion** — wraps each entity with its type-specific markers (e.g. `[CHEMICAL]α-SMA[/CHEMICAL]`)
3. **Tokenization** — tokenizes the marked window with truncation at 512 tokens
4. **Span mask construction** — `build_entity_span_mask` finds the typed marker IDs dynamically per example (not hardcoded to `[E1]`/`[E2]`), then builds a binary mask over the tokens between the start and end marker
5. **Fallback** — if markers are lost after truncation, retokenizes the full text and retries

The mask is used by the model for mention-mean pooling over the entity span.

In [45]:
def insert_entity_markers(text, subject, obj):
    """
    Insert entity marker tokens around subject and object entities.
    
    Args:
        text: Full text
        subject: Subject entity dict with start_idx, end_idx
        obj: Object entity dict with start_idx, end_idx
    
    Returns:
        Text with markers inserted
    """
    # Usa typed markers se disponibili, fallback a generici
    subj_start_m = type_to_marker(subject.get('label', ''))  if subject.get('label') else '[E1]'
    subj_end_m   = type_to_end_marker(subject.get('label', '')) if subject.get('label') else '[/E1]'
    obj_start_m  = type_to_marker(obj.get('label', ''))     if obj.get('label') else '[E2]'
    obj_end_m    = type_to_end_marker(obj.get('label', ''))  if obj.get('label') else '[/E2]'

    # Sort entities by position to insert markers correctly
    entities = [(subject['start_idx'], subject['end_idx'], subj_start_m, subj_end_m),
                (obj['start_idx'], obj['end_idx'], obj_start_m, obj_end_m)]
    entities = sorted(entities, key=lambda x: x[0])
    
    # Insert markers from right to left to maintain positions
    marked_text = text
    offset = 0
    
    for start, end, start_marker, end_marker in entities:
        # Adjust positions with offset
        adj_start = start + offset
        adj_end = end + offset + 1  # +1 because end_idx is inclusive
        
        # Insert markers
        marked_text = (marked_text[:adj_start] + start_marker + 
                      marked_text[adj_start:adj_end] + end_marker + 
                      marked_text[adj_end:])
        
        # Update offset
        offset += len(start_marker) + len(end_marker)
    
    return marked_text


import re

def build_window_around_entities(text, subject, obj, window_chars=300):
    """
    Build a substring window around subject+object to avoid truncation.
    Recomputes subject/object offsets within the window.
    Assumes start/end are inclusive in the original text.
    """
    s_start, s_end = subject["start_idx"], subject["end_idx"]
    o_start, o_end = obj["start_idx"], obj["end_idx"]

    left = min(s_start, o_start)
    right = max(s_end, o_end)

    # expand window
    win_start = max(0, left - window_chars)
    win_end = min(len(text) - 1, right + window_chars)  # inclusive

    window_text = text[win_start:win_end + 1]

    # shift entity indices into window coordinates
    subj_w = dict(subject)
    obj_w = dict(obj)

    subj_w["start_idx"] = s_start - win_start
    subj_w["end_idx"] = s_end - win_start
    obj_w["start_idx"] = o_start - win_start
    obj_w["end_idx"] = o_end - win_start

    # safety clamp
    for ent in (subj_w, obj_w):
        ent["start_idx"] = max(0, min(ent["start_idx"], len(window_text) - 1))
        ent["end_idx"] = max(0, min(ent["end_idx"], len(window_text) - 1))

    return window_text, subj_w, obj_w

def build_entity_span_mask(input_ids, start_token_id, end_token_id):
    """
    Build a mask over the tokens between start marker and end marker (excluded).
    Example: [E1] mention tokens [/E1] -> mask mention tokens only
    """
    input_ids_list = input_ids.tolist()

    try:
        start_idx = input_ids_list.index(start_token_id)
        end_idx = input_ids_list.index(end_token_id)
    except ValueError:
        return None

    if end_idx <= start_idx + 1:
        return None

    mask = torch.zeros_like(input_ids, dtype=torch.long)
    mask[start_idx + 1:end_idx] = 1
    return mask

def tokenize_re_example(
    example,
    tokenizer,
    e1_token_id,
    e1_end_token_id,
    e2_token_id,
    e2_end_token_id,
    max_length=512,
    window_chars=300,
    fallback_to_fulltext=True,
):
    subj_label = example["subject"].get("label", "")
    obj_label  = example["object"].get("label", "")

    # IDs dei marker reali inseriti nel testo (typed se disponibili)
    actual_e1_start = tokenizer.convert_tokens_to_ids(type_to_marker(subj_label)) if subj_label else e1_token_id
    actual_e1_end   = tokenizer.convert_tokens_to_ids(type_to_end_marker(subj_label)) if subj_label else e1_end_token_id
    actual_e2_start = tokenizer.convert_tokens_to_ids(type_to_marker(obj_label)) if obj_label else e2_token_id
    actual_e2_end   = tokenizer.convert_tokens_to_ids(type_to_end_marker(obj_label)) if obj_label else e2_end_token_id

    # Fallback a generici se il typed marker non è nel vocabolario
    unk_id = tokenizer.unk_token_id
    if actual_e1_start == unk_id: actual_e1_start = e1_token_id
    if actual_e1_end   == unk_id: actual_e1_end   = e1_end_token_id
    if actual_e2_start == unk_id: actual_e2_start = e2_token_id
    if actual_e2_end   == unk_id: actual_e2_end   = e2_end_token_id

    w_text, w_subj, w_obj = build_window_around_entities(
        example["text"], example["subject"], example["object"], window_chars=window_chars
    )
    marked_text = insert_entity_markers(w_text, w_subj, w_obj)

    encoding = tokenizer(
        marked_text,
        truncation=True,
        max_length=max_length,
        padding=False,
        return_tensors="pt",
    )
    input_ids      = encoding["input_ids"].squeeze(0)
    attention_mask = encoding["attention_mask"].squeeze(0)

    e1_mask = build_entity_span_mask(input_ids, actual_e1_start, actual_e1_end)
    e2_mask = build_entity_span_mask(input_ids, actual_e2_start, actual_e2_end)

    if (e1_mask is None or e2_mask is None) and fallback_to_fulltext:
        marked_text = insert_entity_markers(example["text"], example["subject"], example["object"])
        encoding = tokenizer(
            marked_text,
            truncation=True,
            max_length=max_length,
            padding="max_length",
            return_tensors="pt",
        )
        input_ids      = encoding["input_ids"].squeeze(0)
        attention_mask = encoding["attention_mask"].squeeze(0)
        e1_mask = build_entity_span_mask(input_ids, actual_e1_start, actual_e1_end)
        e2_mask = build_entity_span_mask(input_ids, actual_e2_start, actual_e2_end)

    if e1_mask is None or e2_mask is None:
        return None
    if e1_mask.sum().item() < 1 or e2_mask.sum().item() < 1:
        return None

    label = label2id[example["predicate"]]
    return {
        "input_ids":      input_ids,
        "attention_mask": attention_mask,
        "e1_mask":        e1_mask,
        "e2_mask":        e2_mask,
        "labels":         torch.tensor(label, dtype=torch.long),
    }

print("Tokenization functions defined")

Tokenization functions defined


In [46]:
# DEBUG: capisce perché tokenize restituisce None
test_example = train_examples[0]

print("Subject:", test_example['subject'])
print("Object: ", test_example['object'])
print()

# Controlla cosa produce insert_entity_markers
w_text, w_subj, w_obj = build_window_around_entities(
    test_example["text"], test_example["subject"], test_example["object"], window_chars=300
)
marked = insert_entity_markers(w_text, w_subj, w_obj)
print("Marked text preview:")
print(marked[:300])
print()

# Controlla quali marker typed vengono inseriti
subj_label = test_example['subject'].get('label', '')
obj_label  = test_example['object'].get('label', '')
print(f"Subject label: '{subj_label}' → marker: '{type_to_marker(subj_label)}'")
print(f"Object label:  '{obj_label}'  → marker: '{type_to_marker(obj_label)}'")
print()

# Tokenizza e cerca i marker
encoding = tokenizer(marked, truncation=True, max_length=512, return_tensors="pt")
input_ids = encoding["input_ids"].squeeze(0)

print(f"e1_token_id={e1_token_id}, e1_end_token_id={e1_end_token_id}")
print(f"e2_token_id={e2_token_id}, e2_end_token_id={e2_end_token_id}")
print()
print(f"[E1]  in input_ids: {(input_ids == e1_token_id).sum().item()}")
print(f"[/E1] in input_ids: {(input_ids == e1_end_token_id).sum().item()}")
print(f"[E2]  in input_ids: {(input_ids == e2_token_id).sum().item()}")
print(f"[/E2] in input_ids: {(input_ids == e2_end_token_id).sum().item()}")

# Cerca anche i typed marker IDs
subj_start_id = tokenizer.convert_tokens_to_ids(type_to_marker(subj_label))
subj_end_id   = tokenizer.convert_tokens_to_ids(type_to_end_marker(subj_label))
obj_start_id  = tokenizer.convert_tokens_to_ids(type_to_marker(obj_label))
obj_end_id    = tokenizer.convert_tokens_to_ids(type_to_end_marker(obj_label))

print(f"\nTyped subj marker '{type_to_marker(subj_label)}' id={subj_start_id}: count={( input_ids == subj_start_id).sum().item()}")
print(f"Typed obj  marker '{type_to_marker(obj_label)}'  id={obj_start_id}: count={(input_ids == obj_start_id).sum().item()}")

Subject: {'start_idx': 1885, 'end_idx': 1889, 'text_span': 'α-SMA', 'label': 'chemical', 'location': 'abstract'}
Object:  {'start_idx': 1919, 'end_idx': 1923, 'text_span': 'colon', 'label': 'anatomical location', 'location': 'abstract'}

Marked text preview:
IBA1, a microglia marker). TDP43 mice treated with butyrate or probiotic VSL#3 had significantly increased rotarod time, increased intestinal mobility and decreased permeability, compared to the untreated group. Butyrate or probiotics treatment decreased the expression of GFAP, TDP43, and increased 

Subject label: 'chemical' → marker: '[CHEMICAL]'
Object label:  'anatomical location'  → marker: '[ANATOMICAL_LOCATION]'

e1_token_id=128027, e1_end_token_id=128028
e2_token_id=128029, e2_end_token_id=128030

[E1]  in input_ids: 0
[/E1] in input_ids: 0
[E2]  in input_ids: 0
[/E2] in input_ids: 0

Typed subj marker '[CHEMICAL]' id=128009: count=1
Typed obj  marker '[ANATOMICAL_LOCATION]'  id=128001: count=1


In [47]:
# Test tokenization
test_example = train_examples[0]
tokenized = tokenize_re_example(
    test_example, tokenizer,
    e1_token_id, e1_end_token_id,
    e2_token_id, e2_end_token_id,
)

print("Test tokenization:")
print(f"  Input IDs shape: {tokenized['input_ids'].shape}")
print(f"  E1 mask tokens: {tokenized['e1_mask'].sum().item()}")
print(f"  E2 mask tokens: {tokenized['e2_mask'].sum().item()}")
print(f"  Label: {tokenized['labels'].item()} ({id2label[tokenized['labels'].item()]})")

# Show marked text
marked = insert_entity_markers(test_example['text'], test_example['subject'], test_example['object'])
print(f"\nMarked text preview: {marked[:200]}...")

Test tokenization:
  Input IDs shape: torch.Size([164])
  E1 mask tokens: 3
  E2 mask tokens: 1
  Label: 12 (located in)

Marked text preview: Probiotics and microbial metabolites maintain barrier and neuromuscular functions and clean protein aggregation to delay disease progression in TDP43 mutation mice. Amyotrophic lateral sclerosis (ALS)...


## Dataset and Pre-tokenization

Pre-tokenizes all examples once and caches them to disk. On subsequent runs, the cache is loaded directly, skipping the tokenization step.

Each item in the dataset contains:
- `input_ids`, `attention_mask` — variable-length tensors (padded to batch max by the collator)
- `e1_mask`, `e2_mask` — binary span masks for subject and object
- `labels` — relation class index
- `sample_weight` — float32 quality weight propagated from the source collection

In [48]:
import torch
from dataclasses import dataclass
from transformers import PreTrainedTokenizerBase

@dataclass
class REDataCollatorWithPadding:
    tokenizer: PreTrainedTokenizerBase
    pad_to_multiple_of: int | None = None

    def __call__(self, features):
        # features: list of dict {input_ids, attention_mask, e1_mask, e2_mask, labels, sample_weight}
        labels = torch.stack([f["labels"] for f in features])
        sample_weight = torch.stack([f["sample_weight"] for f in features]) if "sample_weight" in features[0] else None

        # usa tokenizer.pad per input_ids + attention_mask
        batch = self.tokenizer.pad(
            [{"input_ids": f["input_ids"], "attention_mask": f["attention_mask"]} for f in features],
            padding=True,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )

        # pad manuale per e1/e2_mask alla stessa lunghezza del batch["input_ids"]
        max_len = batch["input_ids"].shape[1]

        def pad_1d(x, pad_value=0):
            # x: tensor [seq_len]
            if x.shape[0] == max_len:
                return x
            out = torch.full((max_len,), pad_value, dtype=x.dtype)
            out[: x.shape[0]] = x
            return out

        e1 = torch.stack([pad_1d(f["e1_mask"]) for f in features])
        e2 = torch.stack([pad_1d(f["e2_mask"]) for f in features])

        batch["e1_mask"] = e1
        batch["e2_mask"] = e2
        batch["labels"] = labels
        if sample_weight is not None:
            batch["sample_weight"] = sample_weight
        return batch


In [49]:
WINDOW_CHARS = 150
# ---- CONFIG CACHE PATHS ----
CACHE_DIR = os.path.join(output_model_dir, "cache_tok")
os.makedirs(CACHE_DIR, exist_ok=True)

TRAIN_CACHE = os.path.join(CACHE_DIR, f"train_dyn_maxlen{max_length}_win{WINDOW_CHARS}.pt")
DEV_CACHE   = os.path.join(CACHE_DIR, f"dev_dyn_maxlen{max_length}_win{WINDOW_CHARS}.pt")

class TensorREDataset(Dataset):
    """Custom dataset for Relation Extraction."""
    
    def __init__(self, tensor_dict):
        self.td = tensor_dict
        self.n = self.td["input_ids"].shape[0]

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        return {
            "input_ids": self.td["input_ids"][idx],
            "attention_mask": self.td["attention_mask"][idx],
            "e1_mask": self.td["e1_mask"][idx],
            "e2_mask": self.td["e2_mask"][idx],
            "labels": self.td["labels"][idx],
        }

from torch.utils.data import Dataset

class ListREDataset(Dataset):
    def __init__(self, items):
        self.items = items
    def __len__(self):
        return len(self.items)
    def __getitem__(self, idx):
        return self.items[idx]

print("Dataset class defined")

def pretokenize_examples_dynamic(
    examples,
    tokenizer,
    e1_token_id,
    e2_token_id,
    max_length=512,
    window_chars=300,
    cache_path=None,
    verbose_every=5000,
):
    if cache_path is not None and os.path.exists(cache_path):
        print(f"[cache] Loading dynamic tokenized dataset from: {cache_path}")
        payload = torch.load(cache_path, map_location="cpu")
        return payload["items"], payload.get("skipped", [])

    print("[cache] Building dynamic tokenized items... (runs once)")
    items = []
    skipped = []

    for i, ex in enumerate(tqdm(examples, desc="Pre-tokenizing(dyn)", total=len(examples))):
        out = None
        try:
            out = tokenize_re_example(
                    ex,
                    tokenizer,
                    e1_token_id,
                    e1_end_token_id,
                    e2_token_id,
                    e2_end_token_id,
                    max_length=max_length,
                    window_chars=window_chars,
                )
        except Exception as e:
            skipped.append((ex.get("pmid"), f"exception:{type(e).__name__}:{str(e)[:120]}"))
            continue

        if out is None:
            skipped.append((ex.get("pmid"), "tokenize_returned_None"))
            continue

        # safety: markers must exist
        if out["e1_mask"].sum().item() < 1 or out["e2_mask"].sum().item() < 1:
                continue

        # ✅ IMPORTANT: out now contains variable-length tensors
        sw = ex.get("sample_weight", 1.0)
        items.append({
            "input_ids":      out["input_ids"].to(torch.int64),
            "attention_mask": out["attention_mask"].to(torch.int64),
            "e1_mask":        out["e1_mask"].to(torch.int64),
            "e2_mask":        out["e2_mask"].to(torch.int64),
            "labels":         out["labels"].to(torch.int64),
            "sample_weight":  torch.tensor(sw, dtype=torch.float32),
        })

        if verbose_every and (i + 1) % verbose_every == 0:
            print(f"  ...processed {i+1}/{len(examples)} | kept={len(items)} | skipped={len(skipped)}")

    print(f"[cache] Done. kept={len(items)} / {len(examples)} | skipped={len(skipped)}")

    if cache_path is not None:
        torch.save({"items": items, "skipped": skipped}, cache_path)
        print(f"[cache] Saved dynamic tokenized dataset to: {cache_path}")

        skip_txt = cache_path.replace(".pt", "_skipped.txt")
        with open(skip_txt, "w", encoding="utf-8") as f:
            for pmid, reason in skipped:
                f.write(f"{pmid}\t{reason}\n")
        print(f"[cache] Saved skipped list to: {skip_txt}")

    return items, skipped


Dataset class defined


In [50]:
WINDOW_CHARS=150

train_items, train_skipped = pretokenize_examples_dynamic(
    train_examples, tokenizer, e1_token_id, e2_token_id,
    max_length=max_length, window_chars=WINDOW_CHARS,
    cache_path=TRAIN_CACHE
)

dev_items, dev_skipped = pretokenize_examples_dynamic(
    dev_examples, tokenizer, e1_token_id, e2_token_id,
    max_length=max_length, window_chars=WINDOW_CHARS,
    cache_path=DEV_CACHE
)

train_dataset = ListREDataset(train_items)
dev_dataset   = ListREDataset(dev_items)

collator = REDataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)


print("FAST datasets ready!")


[cache] Loading dynamic tokenized dataset from: ../../models/deberta_v3_base_re_baseline\cache_tok\train_dyn_maxlen256_win150.pt
[cache] Loading dynamic tokenized dataset from: ../../models/deberta_v3_base_re_baseline\cache_tok\dev_dyn_maxlen256_win150.pt
FAST datasets ready!


## Initialize Model

Instantiates `DeBERTaForREWithEntityMarkers` from `microsoft/deberta-v3-large` and resizes the token embedding matrix to include the newly added typed special tokens.

In [ ]:
# Initialize model
print("Initializing DeBERTa RE model...")
model = DeBERTaForREWithEntityMarkers(model_name, num_labels=len(RELATION_LABELS))

# Resize embeddings per i nuovi special tokens (typed markers)
model.bert.resize_token_embeddings(len(tokenizer))

print(f"Model initialized")
print(f"  Number of labels: {model.num_labels}")
print(f"  Hidden size: {model.bert.config.hidden_size}")

## Custom Trainer

`RETrainer` extends HuggingFace `Trainer` with two changes:

1. **`compute_loss`** — pops `sample_weight` from the batch and passes it to the model forward, enabling quality-aware weighted loss
2. **`_save`** — saves weights as `pytorch_model.bin` instead of safetensors, for compatibility

In [ ]:
import os
import torch
from transformers import Trainer

class RETrainer(Trainer):
    """Custom Trainer that handles entity marker masks + safe saving on Windows."""

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        sample_weight = inputs.pop("sample_weight", None)
        outputs = model(**inputs, labels=labels, sample_weight=sample_weight)
        loss = outputs["loss"]
        return (loss, outputs) if return_outputs else loss

    # ---- CRITICAL PATCH: override _save to avoid safetensors ----
    def _save(self, output_dir: str, state_dict=None):
        os.makedirs(output_dir, exist_ok=True)

        if state_dict is None:
            state_dict = self.model.state_dict()

        # make tensors contiguous (extra-safe)
        for k, v in state_dict.items():
            if isinstance(v, torch.Tensor) and not v.is_contiguous():
                state_dict[k] = v.contiguous()

        # save as classic pytorch bin
        torch.save(state_dict, os.path.join(output_dir, "pytorch_model.bin"))

        # (optional but nice) also save training args
        torch.save(self.args, os.path.join(output_dir, "training_args.bin"))

## Training Arguments

Configuration tuned for `deberta-v3-large`

In [ ]:
import numpy as np
from sklearn.metrics import f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    # Escludi "no relation" (indice 0) dal calcolo F1
    pos_label_ids = [idx for label, idx in label2id.items() if label != "no relation"]

    macro_f1 = f1_score(
        labels, preds,
        labels=pos_label_ids,
        average="macro",
        zero_division=0
    )
    micro_f1 = f1_score(
        labels, preds,
        labels=pos_label_ids,
        average="micro",
        zero_division=0
    )
    return {
        "macro_f1_pos": macro_f1,
        "micro_f1_pos": micro_f1,
    }

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=output_model_dir,

    learning_rate=1e-5,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    max_grad_norm=1.0,

    num_train_epochs=4,
    weight_decay=0.01,

    eval_strategy="steps",
    eval_steps=2000,
    save_strategy="steps",
    save_steps=2000,
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_macro_f1_pos",
    greater_is_better=True,

    fp16=False,
    bf16=torch.cuda.is_available(),
    tf32=False,                        # disabilitato su RTX 5070
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    remove_unused_columns=False,
    disable_tqdm=False,
    logging_steps=50,

    seed=SEED,
    report_to="none",
)

print("Training configuration ready")
print(f"  Model:        {model_name}")
print(f"  LR:           {training_args.learning_rate}")
print(f"  Batch:        {training_args.per_device_train_batch_size} x {training_args.gradient_accumulation_steps} = {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps} effective")
print(f"  Epochs:       {training_args.num_train_epochs}")
print(f"  Best metric:  {training_args.metric_for_best_model}")


## Train Model

In [ ]:
# Initialize trainer
print("Initializing Trainer...")
trainer = RETrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

print("Trainer initialized")
print(f"  Training samples: {len(train_dataset)}")
print(f"  Evaluation samples: {len(dev_dataset)}")

In [ ]:
# Start training
print("="*60)
print("Starting model training...")
print("="*60)

import time
training_start_time = time.time()

train_result = trainer.train()

training_duration = time.time() - training_start_time

print("\n" + "="*60)
print("TRAINING COMPLETED!")
print("="*60)
print(f"Training time: {training_duration/60:.2f} minutes")

## Save Trained Model

In [ ]:
# Save the trained model
print("Saving trained model...")

os.makedirs(output_model_dir, exist_ok=True)
trainer.save_model(output_model_dir)
tokenizer.save_pretrained(output_model_dir)

# Save label mappings
with open(os.path.join(output_model_dir, 'label_mappings.json'), 'w') as f:
    json.dump({'label2id': label2id, 'id2label': id2label}, f, indent=2)

print(f"Model saved to: {output_model_dir}")